In [1]:
user_id = "user_123"
doc_ids = ["medical_doc_001"]

In [2]:
query = "What is my haemoglobin level?"

In [3]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
  model="text-embedding-medembed-large-v0.1",
  base_url="http://127.0.0.1:1234/v1",
  api_key="what-so-ever",
  check_embedding_ctx_length=False
)

In [4]:
import requests

bioMedClip_api_url = "http://localhost:8001/v1/embeddings/text"

In [5]:
text_embedding = embedding_model.embed_query(query)
result = requests.post(
  bioMedClip_api_url,
  json={"text": query},
  timeout=180
)
img_embedding = result.json()["embedding"]

In [6]:
len(text_embedding), len(img_embedding)

(1024, 512)

In [9]:
from qdrant_client import models

COLLECTION_NAME = "medical_documents"

# Only fields that you intend to filter on
PAYLOAD_INDEXES = {
  "user_id": models.PayloadSchemaType.KEYWORD,
  "doc_id": models.PayloadSchemaType.KEYWORD,
  "element_id": models.PayloadSchemaType.KEYWORD,
  "modality": models.PayloadSchemaType.KEYWORD,
  "ingested_at": models.PayloadSchemaType.DATETIME,
}

def initialize_payload_indexes(
  client,
  collection_name=COLLECTION_NAME
):
  # Ensure collection exists
  if not client.collection_exists(collection_name):
    raise ValueError(
      f"Collection '{collection_name}' does not exist. "
      "Create it with your named-vector configuration first."
    )

  # Get existing indexes
  collection_info = client.get_collection(collection_name)
  existing_indexes = collection_info.payload_schema or {}

  # Create only missing indexes
  for field, schema in PAYLOAD_INDEXES.items():
    if field in existing_indexes:
      print(f"Already indexed: {field}")
      continue

    client.create_payload_index(
      collection_name=collection_name,
      field_name=field,
      field_schema=schema,
      wait=True
    )

    print(f"Created index: {field}")

  print("Payload index initialization complete!")

In [10]:
initialize_payload_indexes(
  client=qdrant_client,)

Created index: user_id
Created index: doc_id
Created index: element_id
Created index: modality
Created index: ingested_at
Payload index initialization complete!


In [11]:
from datetime import datetime, timezone
from qdrant_client import models

def hybrid_multimodal_search(
  client,
  collection_name: str,
  text_embedding: list[float],
  biomedclip_text_embedding: list[float],
  user_id: str,
  doc_ids: list[str] | None = None,
  # content_types: list[str] | None = None,
  page_number: int | None = None,
  ingestion_from: datetime | None = None,
  ingestion_to: datetime | None = None,
  top_k: int = 5,
  prefetch_limit: int = 30,
):
  if len(text_embedding) != 1024:
    raise ValueError("Expected 1024D text embedding")

  if len(biomedclip_text_embedding) != 512:
    raise ValueError("Expected 512D BioMedCLIP embedding")

  # Mandatory user filter
  conditions = [
    models.FieldCondition(
      key="user_id",
      match=models.MatchValue(value=user_id)
    )
  ]

  # Filter by multiple document IDs
  if doc_ids:
    conditions.append(
      models.FieldCondition(
        key="doc_id",
        match=models.MatchAny(any=doc_ids)
      )
    )

  # Filter by content type
  # if content_types:
  #   conditions.append(
  #     models.FieldCondition(
  #       key="content_type",
  #       match=models.MatchAny(any=content_types)
  #     )
  #   )

  # Filter by page number
  # if page_number is not None:
  #   conditions.append(
  #     models.FieldCondition(
  #       key="page_number",
  #       match=models.MatchValue(value=page_number)
  #     )
  #   )

  # Ingestion date range
  if ingestion_from or ingestion_to:
    date_range = models.DatetimeRange(
      gte=ingestion_from,
      lte=ingestion_to
    )

    conditions.append(
      models.FieldCondition(
        key="ingested_at",
        range=date_range
      )
    )

  query_filter = models.Filter(must=conditions)

  # Independent searches
  prefetch = [
    models.Prefetch(
      query=text_embedding,
      using="text_vector",
      filter=query_filter,
      limit=prefetch_limit
    ),
    models.Prefetch(
      query=biomedclip_text_embedding,
      using="image_vector",
      filter=query_filter,
      limit=prefetch_limit
    )
  ]

  # Server-side Reciprocal Rank Fusion
  results = client.query_points(
    collection_name=collection_name,
    prefetch=prefetch,
    query=models.FusionQuery(
      fusion=models.Fusion.RRF
    ),
    limit=top_k,
    with_payload=True,
    with_vectors=False
  )

  return results.points


In [12]:
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

load_dotenv()

qdrant_client = QdrantClient(
  url=os.environ.get("QDRANT_URL"),
  api_key=os.environ.get("QDRANT_API_KEY")
)

results = hybrid_multimodal_search(
  client=qdrant_client,
  collection_name="medical_documents",
  text_embedding=text_embedding,
  biomedclip_text_embedding=img_embedding,
  user_id=user_id,
  doc_ids=doc_ids,
  top_k=5,
  prefetch_limit=30
)

In [13]:
results

[ScoredPoint(id='10d56b9f-b976-536a-8c5d-af4bf751f02e', version=7, score=0.5, payload={'user_id': 'user_123', 'doc_id': 'medical_doc_001', 'element_id': '624ae81ef831d57ed30c298d8734bad1_1', 'content_type': 'table', 'modality': 'text', 'source_type': 'unstructured', 'file_name': 'Report.pdf', 'mime_type': 'application/pdf', 'page_number': 1, 'coordinates': {'points': [[141.03102111816406, 621.6292114257812], [141.03102111816406, 2622.771484375], [2442.67333984375, 2622.771484375], [2442.67333984375, 621.6292114257812]], 'system': 'PixelSpace', 'layout_width': 2897, 'layout_height': 4093}, 'text_as_html': '<table><tr><td colspan="4">CBC Haemogram</td></tr><tr><td colspan="4">Investigation Observed Value Unit Biological Reference</td></tr><tr><td colspan="4">Erythrocytes</td></tr><tr><td>Haemoglobin (Hb)</td><td>16.2</td><td>gm/dL</td><td>14-18</td></tr><tr><td>Erythrocyte (RBC) Count</td><td>5.63</td><td>mill/cu.mm</td><td>4.7-6.0</td></tr><tr><td>PCV (Packed Cell Volume)</td><td>48.0</

In [16]:
results[0].payload

{'user_id': 'user_123',
 'doc_id': 'medical_doc_001',
 'element_id': '624ae81ef831d57ed30c298d8734bad1_1',
 'content_type': 'table',
 'modality': 'text',
 'source_type': 'unstructured',
 'file_name': 'Report.pdf',
 'mime_type': 'application/pdf',
 'page_number': 1,
 'coordinates': {'points': [[141.03102111816406, 621.6292114257812],
   [141.03102111816406, 2622.771484375],
   [2442.67333984375, 2622.771484375],
   [2442.67333984375, 621.6292114257812]],
  'system': 'PixelSpace',
  'layout_width': 2897,
  'layout_height': 4093},
 'text_as_html': '<table><tr><td colspan="4">CBC Haemogram</td></tr><tr><td colspan="4">Investigation Observed Value Unit Biological Reference</td></tr><tr><td colspan="4">Erythrocytes</td></tr><tr><td>Haemoglobin (Hb)</td><td>16.2</td><td>gm/dL</td><td>14-18</td></tr><tr><td>Erythrocyte (RBC) Count</td><td>5.63</td><td>mill/cu.mm</td><td>4.7-6.0</td></tr><tr><td>PCV (Packed Cell Volume)</td><td>48.0</td><td>%</td><td>42-52</td></tr><tr><td>MCV (Mean Corpuscular

In [ ]:
from sqlalchemy import select

def attach_retrieved_images(
  points,
  db,
  user_id
):
  image_ids = [point.payload.get("image_id") for point in points if point.payload.get("content_type") == "image"]
  if not image_ids:
    return points  # No images to attach
  
  stmt = (
    select(
      DocumentImage.id,
      DocumentImage.image_data,
    )
    .where(
      DocumentImage.id.in_(image_ids),
      DocumentImage.user_id == user_id
    )
  )

  result = db.execute(stmt)